# Australian ETF Capital Gains Tax (CGT) Calculator

### Overview
This notebook automates the calculation of **CGT** for Australian ETFs.

It imports trade history from a CSV export, tracks individual purchase parcels, applies optimal parcel selection strategies (Highest-Cost-First) and automatically applies the **50% CGT discount** for holdings held over 12 months (365 days).

---

### Key features
* Automatic CSV import; Reads buy and sell confirmations directly from broker exports.
* Chronological auto-sorting: Trades are processed in exact historical order regardless of CSV row order.
* Tax minimisation (HIFO) strategy: Allocates sales against highest cost base parcels first to reduce taxable capital gains.
* ATO myTax ready output: Prints final totals mapped directly to the **Total Current Year Capital Gains** and **Net Capital Gains** fields on myTax.

### Step 1: Define the 'Parcel' Data Structure

A **Parcel** represents a specific batch of units bought on a specific date.
Under ATO TD 33 rules, individual parcels must be tracked separately so their specific cost base and holding duration can be evaluated when sold.

In [1]:
from datetime import datetime, date
from dataclasses import dataclass
import csv

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich import box

# Initialize the rich console
console = Console()

@dataclass
class Parcel:
    """Represents a single purchase parcel of ETF units."""
    parcel_id: int            # Unique ifentifier for the purchase parcel
    ticker: str               # ETF ticker code (e.g. 'IOO', 'NDQ')
    buy_date: date            # Date the parcel was purchased
    remaining_units: float    # Units available for future sale allocation
    current_cost_base: float  # Total cost (Purchase price * Quantity + Brokerage)
    
    @property
    def unit_cost_base(self) -> float:
        """Calculates the cost per individual unit within this parcel"""
        if self.remaining_units <= 0:
            return 0.0
        return self.current_cost_base / self.remaining_units

### Step 2: The 'TaxTracker' Engine

The 'TaxTracker' class handles the core logic:
1. Reads and cleans trade history from the CSV.
2. Sorts all trades chronologically from oldest to newest.
3. Logs buys into individual parcels with brokerage included in the cost base.
4. Processes sells using the HIFO strategy to minimise tax.
5. Applies CGT discounts (50%) for parcels held for at least 12 months.

In [2]:
class TaxTracker:
    def __init__(self):
        # Stores all logged purchase parcels
        self.parcels: list[Parcel] = []
        self._next_id = 1

        # Cumulative financial year totals for myTax lodgement
        self.total_gross_gains_fy = 0.0
        self.total_net_taxable_gains_fy = 0.0
        self.total_proceeds_fy = 0.0

    def _parse_date(self, date_str: str) -> date:
        """Parses various date formats (e.g. 8/06/2022, 08/06/2022, 2022-06-08)."""
        date_str = date_str.strip()
        for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y"):
            try:
                return datetime.strptime(date_str, fmt).date()
            except ValueError:
                pass
        parts = date_str.split('/')
        if len(parts) == 3:
            return date(int(parts[2]), int(parts[1]), int(parts[0]))
        raise ValueError(f"Could not parse date: {date_str}")

    def import_trades_from_csv(self, file_path: str):
        """Reads CSV, sorts all trades chronologically, and processes tax calcs."""
        raw_trades = []
        skipped_count = 0

        # Read raw CSV data and standardise headers
        try:
            with open(file_path, mode='r', encoding='utf-8-sig') as file:
                reader = csv.DictReader(file)
                for line_no, row in enumerate(reader, start=2):
                    clean_row = {k.strip(): v.strip() for k, v in row.items() if k}

                    try:
                        raw_trades.append({
                            'order_type': clean_row['Order Type'].upper(),
                            'ticker': clean_row['AsxCode'],
                            'trade_date': self._parse_date(clean_row['Trade Date']),
                            'units': float(clean_row['Quantity']),
                            'price': float(clean_row['Price']),
                            'brokerage': float(clean_row['Brokerage'])
                        })
                    except (KeyError, ValueError) as e:
                        skipped_count += 1
                        console.print(Panel(
                            f"[bold red]CSV Import Error (Line {line_no}):[/bold red] Invalid or missing field.\n[dim]{e}[/dim]",
                            border_style="red",
                            box=box.ROUNDED,
                            expand=False
                        ))
        except FileNotFoundError:
            console.print(Panel(f"[bold red]FILE NOT FOUND:[/bold red] Could not locate '{file_path}'", border_style="red"))
            return

        # Sort trades chronologically (Oldest to Newest)
        raw_trades.sort(key=lambda t: t['trade_date'])

        buy_count = sum(1 for t in raw_trades if 'BUY' in t['order_type'])
        sell_count = sum(1 for t in raw_trades if 'SELL' in t['order_type'])

        console.print(f"\n[bold italic bright_white]   ETF Capital Gains Tax Breakdown[/bold italic bright_white]")
        console.print(f"[dim]   Loaded {len(raw_trades)} total trades ({buy_count} Buys, {sell_count} Sells) from {file_path}[/dim]\n")

        # Process each trade in timeline sequence
        for trade in raw_trades:
            if 'BUY' in trade['order_type']:
                self.buy_etf(
                    ticker=trade['ticker'],
                    buy_date=trade['trade_date'],
                    units=trade['units'],
                    unit_price=trade['price'],
                    brokerage=trade['brokerage']
                )
            elif 'SELL' in trade['order_type']:
                self.sell_etf(
                    ticker=trade['ticker'],
                    sale_date=trade['trade_date'],
                    units_to_sell=trade['units'],
                    sale_price=trade['price'],
                    brokerage=trade['brokerage']
                )

        # Print final confirmation status
        if skipped_count == 0:
            console.print(f"\n[bold green]✓ All {len(raw_trades)} trades successfully recorded and processed.[/bold green]")
        else:
            console.print(f"\n[bold yellow]⚠ Processed {len(raw_trades)} trades ({skipped_count} invalid rows skipped).[/bold yellow]")

        # Automatically print the final user-friendly myTax summary
        self.print_mytax_lodgement_summary()

    def buy_etf(self, ticker: str, buy_date: date, units: float, unit_price: float, brokerage: float):
        """Logs a new ETF purchase (creates a new parcel)."""
        total_cost_base = (units * unit_price) + brokerage

        parcel = Parcel(
            parcel_id=self._next_id,
            ticker=ticker,
            buy_date=buy_date,
            remaining_units=units,
            current_cost_base=total_cost_base
        )
        self.parcels.append(parcel)
        self._next_id += 1

    def sell_etf(self, ticker: str, sale_date: date, units_to_sell: float, sale_price: float, brokerage: float):
        """Processes a sale, allocates units from active parcels, and calculates CGT with Rich formatting."""
        gross_proceeds = (units_to_sell * sale_price) - brokerage

        # Select active parcels bought on or before sale date
        active_parcels = [
            p for p in self.parcels
            if p.ticker == ticker and p.remaining_units > 0 and p.buy_date <= sale_date
        ]
        total_available = sum(p.remaining_units for p in active_parcels)

        # --- HIGH-VISIBILITY ERROR WARNING FOR SHORTFALLS ---
        if total_available < units_to_sell:
            console.print(Panel(
                f"[bold red]SALE EXECUTION ERROR: Insufficient Units[/bold red]\n\n"
                f"Ticker: [bold blue]{ticker}[/bold blue]\n"
                f"Attempted Sale: [bold white]{units_to_sell:g} units[/bold white] on {sale_date.strftime('%d/%m/%Y')}\n"
                f"Available Owned: [bold red]{total_available:g} units[/bold red]",
                title="[bold red]CGT Calculation Blocked[/bold red]",
                border_style="red",
                box=box.HEAVY,
                expand=False
            ))
            return

        # Sort parcels by highest cost per unit first (HIFO Strategy)
        active_parcels.sort(key=lambda p: p.unit_cost_base, reverse=True)

        remaining_to_sell = units_to_sell
        total_cost_base_used = 0.0
        total_gross_gain = 0.0
        total_taxable_gain = 0.0

        table = Table(
            title=f"\n[italic bright_white]Sale: {units_to_sell:g} units of {ticker} ({sale_date.strftime('%d/%m/%Y')})[/italic bright_white]",
            box=box.HEAVY,
            expand=False,
            padding=(0, 2),
            border_style="bright_black",
            header_style="bold bright_white"
        )
        table.add_column("Parcel Details", justify="left", style="white")
        table.add_column("Net Gain", justify="right")

        for parcel in active_parcels:
            if remaining_to_sell <= 0:
                break

            units_from_parcel = min(parcel.remaining_units, remaining_to_sell)
            fraction_of_parcel = units_from_parcel / parcel.remaining_units

            cost_base_allocated = parcel.current_cost_base * fraction_of_parcel
            proceeds_allocated = (units_from_parcel / units_to_sell) * gross_proceeds
            gross_gain = proceeds_allocated - cost_base_allocated

            # ATO 12-Month Rule Check
            try:
                one_year_later = parcel.buy_date.replace(year=parcel.buy_date.year + 1)
            except ValueError:
                one_year_later = parcel.buy_date.replace(year=parcel.buy_date.year + 1, month=3, day=1)

            is_discounted = sale_date > one_year_later

            if gross_gain > 0 and is_discounted:
                taxable_gain = gross_gain * 0.5
                discount_str = " [dim green](50% CGT)[/dim green]"
            else:
                taxable_gain = gross_gain
                discount_str = ""

            # Deduct sold units from active parcel
            parcel.remaining_units -= units_from_parcel
            parcel.current_cost_base -= cost_base_allocated
            remaining_to_sell -= units_from_parcel

            total_cost_base_used += cost_base_allocated
            total_gross_gain += gross_gain
            total_taxable_gain += taxable_gain

            # Accounting formatting (Red for loss, Green for gain)
            if taxable_gain < 0:
                gain_formatted = f"[bold red]-${abs(taxable_gain):,.2f}[/bold red]"
            else:
                gain_formatted = f"[bold green]${taxable_gain:,.2f}[/bold green]"

            unit_str = "unit" if units_from_parcel == 1 else "units"
            parcel_desc = f"Parcel #{parcel.parcel_id} ({units_from_parcel:g} {unit_str}, bought {parcel.buy_date.strftime('%d/%m/%Y')}){discount_str}"

            table.add_row(
                parcel_desc,
                gain_formatted
            )

        console.print(table)

        # Accumulate annual totals
        self.total_proceeds_fy += gross_proceeds
        self.total_gross_gains_fy += total_gross_gain
        self.total_net_taxable_gains_fy += total_taxable_gain

    def print_mytax_lodgement_summary(self):
        """Prints the final summary formatted cleanly with proper gain/loss color logic."""
        table = Table(
            title="\n[bold italic bright_white]ATO myTax Lodgement Breakdown[/bold italic bright_white]",
            box=box.HEAVY,
            expand=False,
            padding=(0, 2),
            border_style="bright_black",
            header_style="bold bright_white"
        )
        table.add_column("Tax Field", justify="left", style="white")
        table.add_column("Amount", justify="right")

        gross_style = "bold red" if self.total_gross_gains_fy < 0 else "bold green"
        net_style = "bold red" if self.total_net_taxable_gains_fy < 0 else "bold green"

        gross_val = f"-${abs(self.total_gross_gains_fy):,.2f}" if self.total_gross_gains_fy < 0 else f"${self.total_gross_gains_fy:,.2f}"
        net_val = f"-${abs(self.total_net_taxable_gains_fy):,.2f}" if self.total_net_taxable_gains_fy < 0 else f"${self.total_net_taxable_gains_fy:,.2f}"

        table.add_row(
            "Total Current Year Capital Gains (Gross)",
            f"[{gross_style}]{gross_val}[/{gross_style}]"
        )
        table.add_row(
            "Net Capital Gain (Taxable)",
            f"[{net_style}]{net_val}[/{net_style}]"
        )

        console.print(table)

### Step 3: Run the Tax Analysis

Initialise the 'TaxTracker' and load consolidated CSV file ('CMC_Confirmation.csv').

In [3]:
tracker = TaxTracker()
tracker.import_trades_from_csv("CMC_Confirmation.csv")

   ETF Capital Gains Tax Breakdown

   Loaded 16 total trades (14 Buys, 2 Sells) from CMC_Confirmation.csv

                                                                   
                Sale: 14 units of IOO (30/12/2025)                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃  Parcel Details                                    ┃  Net Gain  ┃
┣━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━┫
┃  Parcel #4 (2 units, bought 23/08/2022)            ┃   -$31.31  ┃
┃  Parcel #11 (1 unit, bought 02/01/2025)            ┃    $22.23  ┃
┃  Parcel #12 (2 units, bought 22/01/2025)           ┃    $45.93  ┃
┃  Parcel #8 (1 unit, bought 10/07/2024) (50% CGT)   ┃    $19.05  ┃
┃  Parcel #6 (8 units, bought 28/07/2023) (50% CGT)  ┃   $289.14  ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┻━━━━━━━━━━━━┛

                                                                    
                 Sale: 39 units of NDQ (19/02/2026)                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃  Parcel Details                                     ┃  Net Gain  ┃
┣━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━┫
┃  Parcel #7 (5 units, bought 26/06/2024) (50% CGT)   ┃    $15.42  ┃
┃  Parcel #5 (28 units, bought 28/07/2023) (50% CGT)  ┃   $231.43  ┃
┃  Parcel #3 (6 units, bought 23/08/2022) (50% CGT)   ┃    $66.84  ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┻━━━━━━━━━━━━┛

✓ All 16 trades successfully recorded and processed.

                                                            
               ATO myTax Lodgement Breakdown                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃  Tax Field                                 ┃     Amount  ┃
┣━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━━┫
┃  Total Current Year Capital Gains (Gross)  ┃  $1,280.63  ┃
┃  Net Capital Gain (Taxable)                ┃    $658.74  ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┻━━━━━━━━━━━━━┛